In [ ]:
!pip install -U transformers accelerate peft bitsandbytes datasets sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "/content"  # adapter_model.safetensors + adapter_config.json
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
model = model.merge_and_unload()
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
model.save_pretrained("/content/quantized/model-fp16")
tokenizer.save_pretrained("/content/quantized/model-fp16")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/quantized/model-fp16/tokenizer_config.json',
 '/content/quantized/model-fp16/chat_template.jinja',
 '/content/quantized/model-fp16/tokenizer.json')

In [ ]:
from transformers import BitsAndBytesConfig

int8_config = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config=int8_config,
    device_map="auto"
)

model_int8.save_pretrained("/content/quantized/model-int8")
tokenizer.save_pretrained("/content/quantized/model-int8")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/quantized/model-int8/tokenizer_config.json',
 '/content/quantized/model-int8/chat_template.jinja',
 '/content/quantized/model-int8/tokenizer.json')

In [ ]:
int4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config=int4_config,
    device_map="auto"
)

model_int4.save_pretrained("/content/quantized/model-int4")
tokenizer.save_pretrained("/content/quantized/model-int4")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/quantized/model-int4/tokenizer_config.json',
 '/content/quantized/model-int4/chat_template.jinja',
 '/content/quantized/model-int4/tokenizer.json')

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!pip install -r requirements.txt

Cloning into 'llama.cpp'...
remote: Enumerating objects: 81353, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 81353 (delta 16), reused 4 (delta 4), pack-reused 81319 (from 3)
Receiving objects: 100% (81353/81353), 305.69 MiB | 23.98 MiB/s, done.
Resolving deltas: 100% (58833/58833), done.
/content/llama.cpp
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━

In [ ]:
%cd /content/llama.cpp

/content/llama.cpp


In [ ]:
python convert-hf-to-gguf.py \
  /content/quantized/model-fp16 \
  --outfile /content/quantized/model.gguf \
  --outtype q4_0

SyntaxError: invalid syntax (995414221.py, line 1)

In [ ]:
!python convert/convert-hf-to-gguf.py \
  /content/quantized/model-fp16 \
  --outfile /content/quantized/model.gguf \
  --outtype q4_0

python3: can't open file '/content/llama.cpp/convert/convert-hf-to-gguf.py': [Errno 2] No such file or directory


In [ ]:
!find /content/llama.cpp -name "*gguf*.py"

/content/llama.cpp/convert_hf_to_gguf.py
/content/llama.cpp/convert_lora_to_gguf.py
/content/llama.cpp/convert_hf_to_gguf_update.py
/content/llama.cpp/convert_llama_ggml_to_gguf.py
/content/llama.cpp/examples/model-conversion/scripts/utils/hf-upload-gguf-model.py
/content/llama.cpp/gguf-py/gguf/gguf_reader.py
/content/llama.cpp/gguf-py/gguf/gguf_writer.py
/content/llama.cpp/gguf-py/gguf/scripts/gguf_dump.py
/content/llama.cpp/gguf-py/gguf/scripts/gguf_convert_endian.py
/content/llama.cpp/gguf-py/gguf/scripts/gguf_set_metadata.py
/content/llama.cpp/gguf-py/gguf/scripts/gguf_new_metadata.py
/content/llama.cpp/gguf-py/gguf/scripts/gguf_editor_gui.py
/content/llama.cpp/gguf-py/gguf/scripts/gguf_hash.py
/content/llama.cpp/gguf-py/gguf/gguf.py
/content/llama.cpp/tools/mtmd/legacy-models/minicpmv-convert-image-encoder-to-gguf.py
/content/llama.cpp/tools/mtmd/legacy-models/convert_image_encoder_to_gguf.py
/content/llama.cpp/tools/mtmd/legacy-models/glmedge-convert-image-encoder-to-gguf.py


In [ ]:
%cd /content/llama.cpp

/content/llama.cpp


In [ ]:
!python convert_hf_to_gguf.py \
  /content/quantized/model-fp16 \
  --outfile /content/quantized/model-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: model-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.weig

In [ ]:
!wget https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer.model \
  -O /content/quantized/model-fp16/tokenizer.model

--2026-03-02 13:46:32--  https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer.model
Resolving huggingface.co (huggingface.co)... 18.239.50.103, 18.239.50.16, 18.239.50.49, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.103|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/658fb85235c41262d661dc48/91bf184ab12793d0754344f9095332759432e666320cc6c07f637af50e36db6f?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27tokenizer.model%3B+filename%3D%22tokenizer.model%22%3B&Expires=1772462792&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzcyNDYyNzkyfX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU4ZmI4NTIzNWM0MTI2MmQ2NjFkYzQ4LzkxYmYxODRhYjEyNzkzZDA3NTQzNDRmOTA5NTMzMjc1OTQzMmU2NjYzMjBjYzZjMDdmNjM3YWY1MGUzNmRiNmZcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=B3%7EY1gNxT8%7EpqDJ0WzkxWuztH12pp2pGE4Np5JhXrz

In [ ]:
!ls /content/quantized/model-fp16

chat_template.jinja	model.safetensors      tokenizer.model
config.json		tokenizer_config.json
generation_config.json	tokenizer.json


SyntaxError: invalid syntax (3549840022.py, line 1)

In [ ]:
%cd /content/llama.cpp
!cmake -B build
!cmake --build build --config Release

/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend


In [ ]:
!./build/bin/llama-quantize \
  /content/quantized/model-f16.gguf \
  /content/quantized/model-q4_0.gguf \
  q4_0

main: build = 8185 (2afcdb977)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/quantized/model-f16.gguf' to '/content/quantized/model-q4_0.gguf' as Q4_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from /content/quantized/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Model Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              =

In [ ]:
!./build/bin/llama-quantize \
  /content/quantized/model-f16.gguf \
  /content/quantized/model-q8_0.gguf \
  q8_0

main: build = 8185 (2afcdb977)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/quantized/model-f16.gguf' to '/content/quantized/model-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from /content/quantized/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Model Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              =

In [ ]:
!ls -lh /content/quantized/*.gguf

-rw-r--r-- 1 root root 2.1G Mar  2 13:47 /content/quantized/model-f16.gguf
-rw-r--r-- 1 root root 608M Mar  2 14:15 /content/quantized/model-q4_0.gguf
-rw-r--r-- 1 root root 1.1G Mar  2 14:15 /content/quantized/model-q8_0.gguf


In [ ]:
!du -sh /content/quantized/*

2.1G	/content/quantized/model-f16.gguf
2.1G	/content/quantized/model-fp16
774M	/content/quantized/model-int4
1.2G	/content/quantized/model-int8
608M	/content/quantized/model-q4_0.gguf
1.1G	/content/quantized/model-q8_0.gguf


In [ ]:
mkdir -p /content/day3_deliverables

In [ ]:
!cp -r /content/quantized/model-int8 /content/day3-deliverables/
!cp -r /content/quantized/model-int4 /content/day3-deliverables/
!cp /content/quantized/model-q4_0.gguf /content/day3-deliverables/
!cp /content/quantized/model-q8_0.gguf /content/day3-deliverables/

In [ ]:
mkdir -p /content/day3

In [ ]:
cp -r /content/quantized/model-fp16 \
      /content/quantized/model-int4 \
      /content/quantized/model-int8 \
      /content/quantized/model-f16.gguf \
      /content/quantized/model-q4_0.gguf \
      /content/quantized/model-q8_0.gguf \
      /content/day3

NameError: name 'cp' is not defined

In [ ]:
mkdir -p /content/day3_deliverables && cp -r /content/quantized/model-fp16 /content/quantized/model-int4 /content/quantized/model-int8 /content/quantized/model-f16.gguf /content/quantized/model-q4_0.gguf /content/quantized/model-q8_0.gguf /content/day3_deliverables/

In [ ]:
!zip -r /content/day3_deliverables.zip /content/day3_deliverables

  adding: content/day3_deliverables/ (stored 0%)
  adding: content/day3_deliverables/model-int8/ (stored 0%)
  adding: content/day3_deliverables/model-int8/generation_config.json (deflated 29%)
  adding: content/day3_deliverables/model-int8/config.json (deflated 56%)
  adding: content/day3_deliverables/model-int8/tokenizer_config.json (deflated 46%)
  adding: content/day3_deliverables/model-int8/chat_template.jinja (deflated 60%)
  adding: content/day3_deliverables/model-int8/model.safetensors (deflated 14%)
  adding: content/day3_deliverables/model-int8/tokenizer.json (deflated 85%)
  adding: content/day3_deliverables/model-f16.gguf (deflated 20%)
  adding: content/day3_deliverables/model-q4_0.gguf (deflated 5%)
  adding: content/day3_deliverables/model-int4/ (stored 0%)
  adding: content/day3_deliverables/model-int4/generation_config.json (deflated 29%)
  adding: content/day3_deliverables/model-int4/config.json (deflated 56%)
  adding: content/day3_deliverables/model-int4/tokenizer_c

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp /content/day3_deliverables.zip "/content/drive/MyDrive/"
print("✅ Successfully copied to Google Drive!")

✅ Successfully copied to Google Drive!
